# RetailPulse - Day 13

## Automated Retraining Pipeline with Airflow

Objective:
Define the retraining pipeline logic (drift check -> conditional retrain -> evaluate -> save) as standalone Python functions, validate them in this notebook, then wrap them into an Airflow DAG (`retraining_dag.py`) for orchestration.

Note: Airflow itself is a separate orchestration service (a scheduler + webserver), not something that runs inside a Jupyter notebook. This notebook builds and tests the pipeline functions; the generated `retraining_dag.py` file is what would be placed in an Airflow `dags/` folder in production.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

import xgboost as xgb

from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset

import json
import os
from datetime import datetime

## Step 1: Define Feature Columns and Paths

In [2]:
FEATURE_COLS = [
    "Recency",
    "Frequency",
    "Monetary",
    "AvgOrderValue",
    "UniqueProducts",
    "TotalQuantity",
    "AvgCustomerFrequency"
]

TARGET_COL = "Churn"

DATA_PATH = "../data/churn_predictions_tuned.csv"

MODEL_PATH = "../models/xgb_churn_model_tuned.json"

RETRAIN_MODEL_PATH = "../models/xgb_churn_model_retrained.json"

DRIFT_LOG_PATH = "../data/retraining_log.csv"

RETRAIN_THRESHOLD = 0.5

MIN_AUC_THRESHOLD = 0.80

## Step 2: Pipeline Function - Check Drift

Reuses the Day 12 drift simulation/check logic, returning whether retraining is recommended.

In [3]:
def check_drift(data_path=DATA_PATH, retrain_threshold=RETRAIN_THRESHOLD, seed=42):

    df = pd.read_csv(data_path)

    model_cols = FEATURE_COLS + ["ChurnProbability_Tuned"]

    features = df[model_cols].fillna(0).reset_index(drop=True)

    split_point = len(features) // 2

    reference = features.iloc[:split_point].copy()
    current = features.iloc[split_point:].copy()

    rng = np.random.RandomState(seed)

    current["Recency"] = current["Recency"] * rng.uniform(1.15, 1.35, len(current))
    current["Monetary"] = current["Monetary"] * rng.uniform(0.75, 0.95, len(current))
    current["AvgOrderValue"] = current["AvgOrderValue"] * rng.uniform(0.8, 1.0, len(current))

    dd = DataDefinition()

    ref_ds = Dataset.from_pandas(reference, data_definition=dd)
    cur_ds = Dataset.from_pandas(current, data_definition=dd)

    report = Report(metrics=[DataDriftPreset()])

    result = report.run(cur_ds, ref_ds)

    rd = result.dict()

    drifted_count_metric = next(
        m for m in rd["metrics"]
        if "DriftedColumnsCount" in m["metric_name"]
    )

    drift_share = drifted_count_metric["value"]["share"]

    retrain_needed = drift_share >= retrain_threshold

    return {
        "drift_share": drift_share,
        "retrain_needed": retrain_needed
    }

In [4]:
drift_result = check_drift()

drift_result

{'drift_share': 0.125, 'retrain_needed': False}

## Step 3: Pipeline Function - Retrain Model

In [5]:
def retrain_model(data_path=DATA_PATH, model_out_path=RETRAIN_MODEL_PATH):

    df = pd.read_csv(data_path)

    X = df[FEATURE_COLS].fillna(0)
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    smote = SMOTE(random_state=42)

    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        random_state=42
    )

    model.fit(X_train_res, y_train_res)

    proba = model.predict_proba(X_test)[:,1]

    auc = roc_auc_score(y_test, proba)

    os.makedirs(os.path.dirname(model_out_path), exist_ok=True)

    model.save_model(model_out_path)

    return {
        "auc": auc,
        "model_path": model_out_path
    }

## Step 4: Pipeline Function - Evaluate and Decide Promotion

Compares retrained model AUC against the minimum acceptable threshold to decide whether to promote it (replace the production model).

In [6]:
def evaluate_and_promote(retrain_result, min_auc=MIN_AUC_THRESHOLD, promote=True):

    auc = retrain_result["auc"]

    passed = auc >= min_auc

    promoted = False

    if passed and promote:

        import shutil

        shutil.copy(retrain_result["model_path"], MODEL_PATH)

        promoted = True

    return {
        "auc": auc,
        "min_auc_threshold": min_auc,
        "passed_threshold": passed,
        "promoted": promoted
    }

## Step 5: Pipeline Function - Log Run

In [7]:
def log_pipeline_run(drift_info, retrain_info=None, eval_info=None, log_path=DRIFT_LOG_PATH):

    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "drift_share": drift_info["drift_share"],
        "retrain_triggered": drift_info["retrain_needed"],
        "new_model_auc": retrain_info["auc"] if retrain_info else None,
        "min_auc_threshold": eval_info["min_auc_threshold"] if eval_info else None,
        "passed_threshold": eval_info["passed_threshold"] if eval_info else None,
        "model_promoted": eval_info["promoted"] if eval_info else None
    }

    log_df = pd.DataFrame([log_entry])

    if os.path.exists(log_path):

        existing = pd.read_csv(log_path)

        log_df = pd.concat([existing, log_df], ignore_index=True)

    log_df.to_csv(log_path, index=False)

    return log_entry

## Step 6: Run the Full Pipeline End-to-End (Local Simulation)

This simulates exactly what the Airflow DAG will execute: check drift -> if drift detected, retrain -> evaluate -> promote if passing -> log results.

In [8]:
drift_info = check_drift()

retrain_info = None
eval_info = None

if drift_info["retrain_needed"]:

    print("Drift detected (share = {:.2f}). Retraining model...".format(drift_info["drift_share"]))

    retrain_info = retrain_model()

    print("Retrained model AUC:", retrain_info["auc"])

    eval_info = evaluate_and_promote(retrain_info)

    print("Evaluation result:", eval_info)

else:

    print("No significant drift detected (share = {:.2f}). Skipping retraining.".format(drift_info["drift_share"]))

log_entry = log_pipeline_run(drift_info, retrain_info, eval_info)

log_entry

No significant drift detected (share = 0.12). Skipping retraining.


{'timestamp': '2026-06-13T12:10:07.047709',
 'drift_share': 0.125,
 'retrain_triggered': False,
 'new_model_auc': None,
 'min_auc_threshold': None,
 'passed_threshold': None,
 'model_promoted': None}

In [9]:
retraining_log = pd.read_csv(DRIFT_LOG_PATH)

retraining_log

,timestamp,drift_share,retrain_triggered,new_model_auc,min_auc_threshold,passed_threshold,model_promoted
0,2026-06-13T12:10:07.047709,0.125,False,NaN,NaN,NaN,NaN


## Step 7: Export Pipeline Functions to a Reusable Module

We write the pipeline functions to `pipeline_tasks.py` so the Airflow DAG can import and call them as task functions.

In [10]:
pipeline_code = '''"""
RetailPulse retraining pipeline task functions.
Used by retraining_dag.py (Airflow) and by Day13 notebook for local testing.
"""

import pandas as pd
import numpy as np
import os
import shutil
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb

from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset


FEATURE_COLS = [
    "Recency", "Frequency", "Monetary", "AvgOrderValue",
    "UniqueProducts", "TotalQuantity", "AvgCustomerFrequency"
]
TARGET_COL = "Churn"

DATA_PATH = "data/churn_predictions_tuned.csv"
MODEL_PATH = "models/xgb_churn_model_tuned.json"
RETRAIN_MODEL_PATH = "models/xgb_churn_model_retrained.json"
DRIFT_LOG_PATH = "data/retraining_log.csv"

RETRAIN_THRESHOLD = 0.5
MIN_AUC_THRESHOLD = 0.80


def check_drift(data_path=DATA_PATH, retrain_threshold=RETRAIN_THRESHOLD, seed=42):

    df = pd.read_csv(data_path)

    model_cols = FEATURE_COLS + ["ChurnProbability_Tuned"]

    features = df[model_cols].fillna(0).reset_index(drop=True)

    split_point = len(features) // 2

    reference = features.iloc[:split_point].copy()
    current = features.iloc[split_point:].copy()

    rng = np.random.RandomState(seed)

    current["Recency"] = current["Recency"] * rng.uniform(1.15, 1.35, len(current))
    current["Monetary"] = current["Monetary"] * rng.uniform(0.75, 0.95, len(current))
    current["AvgOrderValue"] = current["AvgOrderValue"] * rng.uniform(0.8, 1.0, len(current))

    dd = DataDefinition()

    ref_ds = Dataset.from_pandas(reference, data_definition=dd)
    cur_ds = Dataset.from_pandas(current, data_definition=dd)

    report = Report(metrics=[DataDriftPreset()])

    result = report.run(cur_ds, ref_ds)

    rd = result.dict()

    drifted_count_metric = next(
        m for m in rd["metrics"]
        if "DriftedColumnsCount" in m["metric_name"]
    )

    drift_share = drifted_count_metric["value"]["share"]

    return {
        "drift_share": drift_share,
        "retrain_needed": drift_share >= retrain_threshold
    }


def retrain_model(data_path=DATA_PATH, model_out_path=RETRAIN_MODEL_PATH):

    df = pd.read_csv(data_path)

    X = df[FEATURE_COLS].fillna(0)
    y = df[TARGET_COL]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="auc", random_state=42
    )

    model.fit(X_train_res, y_train_res)

    proba = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, proba)

    os.makedirs(os.path.dirname(model_out_path), exist_ok=True)
    model.save_model(model_out_path)

    return {"auc": auc, "model_path": model_out_path}


def evaluate_and_promote(retrain_result, min_auc=MIN_AUC_THRESHOLD, promote=True):

    auc = retrain_result["auc"]
    passed = auc >= min_auc
    promoted = False

    if passed and promote:
        shutil.copy(retrain_result["model_path"], MODEL_PATH)
        promoted = True

    return {
        "auc": auc,
        "min_auc_threshold": min_auc,
        "passed_threshold": passed,
        "promoted": promoted
    }


def log_pipeline_run(drift_info, retrain_info=None, eval_info=None, log_path=DRIFT_LOG_PATH):

    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "drift_share": drift_info["drift_share"],
        "retrain_triggered": drift_info["retrain_needed"],
        "new_model_auc": retrain_info["auc"] if retrain_info else None,
        "min_auc_threshold": eval_info["min_auc_threshold"] if eval_info else None,
        "passed_threshold": eval_info["passed_threshold"] if eval_info else None,
        "model_promoted": eval_info["promoted"] if eval_info else None
    }

    log_df = pd.DataFrame([log_entry])

    if os.path.exists(log_path):
        existing = pd.read_csv(log_path)
        log_df = pd.concat([existing, log_df], ignore_index=True)

    log_df.to_csv(log_path, index=False)

    return log_entry
'''

with open("../pipeline_tasks.py", "w") as f:
    f.write(pipeline_code)

print("pipeline_tasks.py written to project root")

pipeline_tasks.py written to project root


## Step 8: Generate the Airflow DAG File

This DAG defines a daily pipeline with four tasks: `check_drift_task` -> `retrain_task` (conditional) -> `evaluate_task` -> `log_task`. It imports the functions from `pipeline_tasks.py`.

To use in production: place both `pipeline_tasks.py` and `retraining_dag.py` in your Airflow `dags/` folder (with `data/` and `models/` accessible at the same relative paths, or update the paths).

In [11]:
dag_code = '''"""
RetailPulse - Automated Retraining DAG

Daily pipeline:
1. check_drift_task   - run Evidently drift report on latest churn data
2. retrain_task       - retrain XGBoost churn model (only if drift detected)
3. evaluate_task       - compare new model AUC against minimum threshold, promote if passing
4. log_task           - append run results to retraining_log.csv

Place this file (and pipeline_tasks.py) in the Airflow dags/ folder.
"""

from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.empty import EmptyOperator
from datetime import datetime, timedelta

from pipeline_tasks import (
    check_drift,
    retrain_model,
    evaluate_and_promote,
    log_pipeline_run
)


default_args = {
    "owner": "retailpulse",
    "retries": 1,
    "retry_delay": timedelta(minutes=5)
}


def _check_drift(**context):

    result = check_drift()

    context["ti"].xcom_push(key="drift_info", value=result)

    return "retrain_task" if result["retrain_needed"] else "skip_retrain_task"


def _retrain(**context):

    result = retrain_model()

    context["ti"].xcom_push(key="retrain_info", value=result)


def _evaluate(**context):

    retrain_info = context["ti"].xcom_pull(key="retrain_info", task_ids="retrain_task")

    result = evaluate_and_promote(retrain_info)

    context["ti"].xcom_push(key="eval_info", value=result)


def _log(**context):

    drift_info = context["ti"].xcom_pull(key="drift_info", task_ids="check_drift_task")

    retrain_info = context["ti"].xcom_pull(key="retrain_info", task_ids="retrain_task")

    eval_info = context["ti"].xcom_pull(key="eval_info", task_ids="evaluate_task")

    log_pipeline_run(drift_info, retrain_info, eval_info)


with DAG(
    dag_id="retailpulse_churn_retraining",
    description="Drift-triggered retraining pipeline for RetailPulse churn model",
    default_args=default_args,
    schedule="@daily",
    start_date=datetime(2026, 1, 1),
    catchup=False,
    tags=["retailpulse", "mlops", "churn"]
) as dag:

    check_drift_task = BranchPythonOperator(
        task_id="check_drift_task",
        python_callable=_check_drift
    )

    retrain_task = PythonOperator(
        task_id="retrain_task",
        python_callable=_retrain
    )

    skip_retrain_task = EmptyOperator(
        task_id="skip_retrain_task"
    )

    evaluate_task = PythonOperator(
        task_id="evaluate_task",
        python_callable=_evaluate
    )

    log_task = PythonOperator(
        task_id="log_task",
        python_callable=_log,
        trigger_rule="none_failed_min_one_success"
    )

    check_drift_task >> [retrain_task, skip_retrain_task]
    retrain_task >> evaluate_task >> log_task
    skip_retrain_task >> log_task
'''

with open("../retraining_dag.py", "w") as f:
    f.write(dag_code)

print("retraining_dag.py written to project root")

retraining_dag.py written to project root


## Step 9: Visualize the Pipeline (Architecture Diagram in Text)

In [12]:
pipeline_diagram = '''
RetailPulse - Automated Retraining Pipeline (Daily Schedule)

  +----------------------+
  |  check_drift_task    |
  |  (Evidently AI)       |
  +----------+-----------+
             |
     drift >= threshold?
        /          \\
      yes            no
       |              |
  +----v-----+   +----v-------------+
  | retrain_ |   | skip_retrain_task |
  | task     |   +----+-------------+
  | (XGBoost)|        |
  +----+-----+        |
       |               |
  +----v-----+         |
  | evaluate_|         |
  | task     |         |
  | (AUC check,        |
  |  promote model)    |
  +----+-----+         |
       |               |
       +-------+-------+
               |
        +------v-------+
        |  log_task     |
        | (retraining_  |
        |  log.csv)      |
        +---------------+
'''

print(pipeline_diagram)


RetailPulse - Automated Retraining Pipeline (Daily Schedule)

  +----------------------+
  |  check_drift_task    |
  |  (Evidently AI)       |
  +----------+-----------+
             |
     drift >= threshold?
        /          \
      yes            no
       |              |
  +----v-----+   +----v-------------+
  | retrain_ |   | skip_retrain_task |
  | task     |   +----+-------------+
  | (XGBoost)|        |
  +----+-----+        |
       |               |
  +----v-----+         |
  | evaluate_|         |
  | task     |         |
  | (AUC check,        |
  |  promote model)    |
  +----+-----+         |
       |               |
       +-------+-------+
               |
        +------v-------+
        |  log_task     |
        | (retraining_  |
        |  log.csv)      |
        +---------------+



# Day 13 Findings

1. Defined the retraining pipeline as four standalone functions: `check_drift`, `retrain_model`, `evaluate_and_promote`, and `log_pipeline_run`.
2. `check_drift` reuses the Day 12 Evidently AI drift simulation and returns whether retraining is needed.
3. `retrain_model` retrains the XGBoost churn model (Day 9/11 configuration) with SMOTE balancing and returns the new model's AUC.
4. `evaluate_and_promote` compares the new model's AUC against a minimum threshold (0.80) and promotes it to production (overwrites the tuned model file) only if it passes.
5. `log_pipeline_run` appends each pipeline run's results to `retraining_log.csv` for auditability.
6. Ran the full pipeline end-to-end locally in this notebook to validate the logic before deployment.
7. Exported the pipeline functions to `pipeline_tasks.py` and generated `retraining_dag.py`, an Airflow DAG using `BranchPythonOperator` to conditionally retrain only when drift is detected, scheduled to run daily.
8. Documented the pipeline architecture as a flow diagram for inclusion in project documentation.